In [1]:
!pip install --upgrade transformers accelerate==0.22.0 einops==0.6.1 langchain==0.0.300 xformers==0.0.21
!pip install bitsandbytes==0.41.1 chromadb==0.4.12 datasets accelerate --upgrade huggingface_hub
!pip install torchvision==0.4.0 -f https://download.pytorch.org/whl/torch_stable.html
!pip install -U sentence-transformers pypdf2 pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.0/167.0 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 64.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 61.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 82.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys
from torch import cuda, bfloat16
import torch
import transformers
import chromadb
from chromadb.config import Settings
from langchain.llms import HuggingFacePipeline
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma

In [3]:
model_id = '/kaggle/input/gemma/transformers/2b-it/3'

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer 

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c107WarningC1ENS_7variantIJNS0_11UserWarningENS0_18DeprecationWarningEEEERKNS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEEb'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.t

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:

query_pipeline = transformers.pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto",
        max_length = 10000,
)

Device set to use cuda:0


In [6]:
import pandas as pd 

df = pd.read_json('/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)/queries.json')

df.head()

,query_id,question
0,1,What is the scientific method in psychology?
1,2,What are the basic parts of a neuron?
2,3,What are the stages of sleep?
3,4,What is operant conditioning?
4,5,What is problem-solving in psychology?


In [7]:
llm = HuggingFacePipeline(pipeline=query_pipeline)

In [8]:
from PyPDF2 import PdfReader
import pdfplumber

def load_book_with_page_overlap(pdf_path, overlap_size=0):
    """Load and preprocess the book from PDF with word overlap between pages."""
    sections = []
    # Function to add overlap between two pages
    def overlap_pages(page_text, next_page_text, overlap_size):
        page_words = page_text.split()
        next_page_words = next_page_text.split()
        # Take the last `overlap_size` words from the current page
        overlap = page_words[-overlap_size:] if len(page_words) > overlap_size else page_words
        combined_text = " ".join(overlap) + " " + " ".join(next_page_words)
        return combined_text
    try:
        # Try loading with PyPDF2
        reader = PdfReader(pdf_path, strict=False)  # Ignore structural issues
        previous_page_text = None  # To hold the last page's text for overlap
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                if previous_page_text:
                # Overlap the current page with the previous one
                    combined_text = overlap_pages(previous_page_text, text, overlap_size)
                    sections.append({"page": page_num + 1, "text": combined_text.strip()})
                else:
                    sections.append({"page": page_num + 1, "text": text.strip()})
                # Update `previous_page_text` to be the current page's text for the next iteration
                previous_page_text = text.strip()
    except Exception as e:
        print(f"Error loading PDF with PyPDF2: {e}")
        print("Falling back to pdfplumber...")
        try:
            # Use pdfplumber as a fallback
            with pdfplumber.open(pdf_path) as pdf:
                previous_page_text = None
                for page_num, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text and text.strip():
                        if previous_page_text:
                            # Overlap the current page with the previous one
                            combined_text = overlap_pages(previous_page_text, text, overlap_size)
                            sections.append({"page": page_num + 1, "text": combined_text.strip()})
                        else:
                            sections.append({"page": page_num + 1, "text": text.strip()})
                        # Update `previous_page_text` for the next iteration
                        previous_page_text = text.strip()
        except Exception as fallback_error:
            print(f"Error loading PDF with pdfplumber: {fallback_error}")
            return None
    return sections

pdf_path = '/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)/book.pdf'
book_data= load_book_with_page_overlap(pdf_path, overlap_size=5)

if book_data:
    print(f"Loaded {len(book_data)} pages successfully.")
else:
    print("Failed to load the PDF.")

Loaded 751 pages successfully.


In [9]:

for i, section in enumerate(book_data[:5]):  # Show first 5 sections

    print(f"Page {section['page']} - Text: {section['text'][:150]}...")  

Page 3 - Text: Psychology 2e          SENIOR CONTRIBUTING AUTHORS ROSE M. SPIELMAN, FORMERLY OF QUINNIPIAC UNIVERSITY WILLIAM J. JENKINS, MERCER UNIVERSITY MARILYN D...
Page 4 - Text: MARILYN D. LOVETT, SPELMAN COLLEGE OpenStax Rice University 6100 Main Street MS-375 Houston, Texas 77005 To learn more about OpenStax, visit https://o...
Page 5 - Text: 9 10 JAY 22 20 OPENSTAX OpenStax provides free, peer-reviewed, openly licensed textbooks for introductory college and Advanced Placement® courses and ...
Page 6 - Text: Futures William Marsh Rice University Study where you want, what you want, when you want. Access. The future of education. openstax.org When you acces...
Page 7 - Text: forever. Get started at openstax.org/details/books/psychology-2e CONTENT S Prefac e 1 CHAPTER 1 Introduction t o Psycholog y 7 Intro duction 7 1.1Wha ...


In [10]:
from datasets import Dataset
dataset = Dataset.from_list(book_data)
print(dataset)

Dataset({
    features: ['page', 'text'],
    num_rows: 751
})


In [11]:
model_kwargs = {"device": "cuda"}
local_model_path = "/kaggle/input/sentence-transformers/minilm-l6-v2/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=local_model_path, model_kwargs=model_kwargs)

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [12]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content=section['text'], 
        metadata={"page": section['page']},
    ) 
    for section in book_data
]

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# Split the documents into smaller chunks
all_splits = text_splitter.split_documents(documents)

In [13]:
vectordb = Chroma.from_documents(documents=all_splits, embedding=embeddings, persist_directory="chroma_db")

Batches:   0%|          | 0/95 [00:00<?, ?it/s]

In [14]:
retriever = vectordb.as_retriever()

qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)

In [15]:
import csv
import json

def query_and_retrieve(query,query_id, vectordb):
    """
    Perform a precision-based retrieval and generate a structured answer.

    Args:
        query (str): The query string.
        vectordb: The vector database object to perform similarity searches.

    Returns:
        dict: The final structured answer and references.
    """
    # Stage 1: Core concept retrieval
    concept_results = vectordb.similarity_search(
        query=f"definition explanation {query}",
        k=1000   # Number of results to retrieve
    )

    # Stage 2: Supporting detail retrieval
    detail_results = vectordb.similarity_search(
        query=f"example application {query}",
        k=1000
    )

    # Combine and prioritize contexts
    contexts = []
    pages = set()

    # Process concept_results and detail_results
    for results in [concept_results, detail_results]:
        for doc in results:
            contexts.append(doc.page_content)
            pages.add(str(doc.metadata.get('page', 'Unknown')))

    # Combine contexts and prepare the input for the QA system
    combined_context = ' '.join(contexts[:2])  # Limit to top 2 relevant contexts
    sources = list(sorted(pages))[:3]  # Limit references to top 3 pages

    # Generate structured answer using the combined context
    prompt = f""" refer context and answer


### Input:
**Context:**  
{combined_context}

**Question:**  
{query}

### Output:
**Answer:**  

"""
    # Call the QA system to generate the answer
    try:
        result = qa.run(prompt)
    except Exception as e:
        print(f"Error during model execution: {e}")
        result = "Error generating answer"

    # Map the query to a precise section
    section = map_to_precise_section(query, combined_context)

    # Format the response
    return {
        'ID': query_id,
        'context': combined_context,
        'answer': result,
        'references': json.dumps({
            'sections': [section],
            'pages': sources
        })
    }


def create_submission_file(queries_df, vector_store):
    """
    Create the submission file based on the provided queries and vector store.

    Args:
        queries_df (pd.DataFrame): DataFrame containing the test set queries and IDs.
        vector_store: The vector database for document retrieval.
    """
    results = []

    for _, row in queries_df.iterrows():
        query_id = row['query_id']
        query = row['question']

        # Generate structured answers for each query
        result = query_and_retrieve(query,query_id, vector_store)
        results.append(result)

    # Convert results to a DataFrame for submission
    submission_df = pd.DataFrame(results)
    submission_df.to_csv('submission.csv', index=False)
    print("Submission file saved as 'submission.csv'")

def map_to_precise_section(query, context):
    """
    Map the query to the most relevant section based on context and query keywords.
    """
    section_mapping = {
        'scientific': 'psychological_research/approaches_to_research',
        'brain': 'biopsychology/the_brain_and_behavior',
        'memory': 'memory/how_memory_functions',
        'development': 'developmental_psychology/introduction',
        'personality': 'personality/introduction_to_personality',
        'emotion': 'emotion_and_motivation/emotions',
        'learning': 'learning/introduction',
        'social': 'social_psychology/introduction'
    }

    for key, section in section_mapping.items():
        if key in query.lower() or key in context.lower():
            return section
    return "introduction_to_psychology"


In [16]:
create_submission_file(df, vectordb)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]



> Entering new RetrievalQA chain...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]



> Entering new RetrievalQA chain...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


> Finished chain.
Submission file saved as 'submission.csv'
